# Initialize the Client

As a first step, you need to initialize the client to interact with the Aignostics Platform. This will execute an OAuth flow depending on the environment you run:
- In case you have a browser available, an interactive login flow in your browser is started.
- In case there is no browser available, a device flow is started.

**NOTE:** By default, the client caches the access token in your operation systems application cache folder. If you do not want to store the access token, please initialize the client like this:

```python
client = aignostics.client.Client(cache_token=False)
```



In [1]:
from collections.abc import Iterator

import pandas as pd
from pydantic import BaseModel


# the following function is used for visualizing the results nicely in this notebook
def show(models: BaseModel | list[BaseModel] | Iterator[BaseModel]) -> pd.DataFrame:
    """Visualize the results in a pandas DataFrame.

    Returns:
        pd.DataFrame: A DataFrame containing the results.
    """
    items = [models.model_dump()] if isinstance(models, BaseModel) else (a.model_dump() for a in models)
    return pd.DataFrame(items)

In [1]:
import aignostics.client

# initialize the client
client = aignostics.client.Client()

# List our available applications

Next, let us list the applications that are available in your organization:

In [3]:
applications = client.applications.list()
# visualize
show(applications)

,application_id,name,slug,regulatory_classes,description
0,ee5566d2-d3cb-4303-9e23-8a5ab3e5b8ed,TwoTaskDummy,two-task-dummy,"[demo, RuO]",This is just a dummy application with two algo...
1,f7aa7f53-3b4c-476a-bc25-561ef9cfbf6d,H&E TME,h-e-tme,[],This is the H&E TME Application.


# List all available versions of an application

Now that we know the applications that are available, we can list all the versions of a specific application. In this case, we will use the `TwoTask Dummy Application` as an example, which has the `application_id`: `ee5566d2-d3cb-4303-9e23-8a5ab3e5b8ed`. Using the `application_id`, we can list all the versions of the application:

In [4]:
application_versions = client.applications.versions.list(for_application="ee5566d2-d3cb-4303-9e23-8a5ab3e5b8ed")
# visualize
show(application_versions)

,application_version_id,application_version_slug,version,application_id,flow_id,changelog,input_artifacts,output_artifacts
0,60e7b441-307a-4b41-8a97-5b02e7bc73a4,two-task-dummy:v0.0.3,0.0.3,ee5566d2-d3cb-4303-9e23-8a5ab3e5b8ed,None,<some_changelog>,"[{'name': 'user_slide', 'mime_type': 'image/ti...",[{'name': 'tissue_segmentation:geojson_polygon...


# Inspect the application version details

Now that we have the list of versions, we can inspect the details of a specific version. While we could directly use the list of application version returned by the `list` method, we want to directly query details for a specific application version. In this case, we will use version `0.0.3`, which has the `application_version_id`: `60e7b441-307a-4b41-8a97-5b02e7bc73a4`. We use the `application_version_id` to retrieve further details about the application version:

In [5]:
from IPython.display import JSON

# get the application version details
two_task_app = client.applications.versions.details(for_application_version_id="60e7b441-307a-4b41-8a97-5b02e7bc73a4")

# view the `input_artifacts` to get insights in the required fields of the application version payload
JSON(two_task_app.input_artifacts[0].to_json())

/Users/akunft/dev/python-sdk/.venv/lib/python3.11/site-packages/IPython/core/display.py:636: UserWarning: JSON expects JSONable dict or list, not JSON strings
  warnings.warn("JSON expects JSONable dict or list, not JSON strings")


<IPython.core.display.JSON object>

# Trigger an application run

Now, let's trigger an application run for the `TwoTask Dummy Application`. We will use the `application_version_id` that we retrieved in the previous step. To create an application run, we need to provide a payload that consists of 1 or more `Items`. We provide the Pydantic model `ItemCreationRequest` an item and the data that comes with it:
```python
ItemCreationRequest(
    reference="<a unique reference associate outputs to this input item>",
    input_artifacts=[InputArtifactCreationRequest]
)
```
The `InputArtifactCreationRequest` defines the actual data that you provide aka. in this case the image that you want to be processed. The expected values are defined by the application version and have to align with the `input_artifacts` schema of the application version. In the case of the two task dummy application, we only require a single artifact per item, which is the image to process on. The artifact name is defined as `user_slide`. The `download_url` is a signed URL that allows the Aignostics Platform to download the image data later during processing. In addition to the image data itself, you have to provide the metadata defined in the input artifact schema, i.e., `checksum_crc32c`, `base_mpp`, `width`, and `height`. The metadata is used to validate the input data and is required for the processing of the image. The following example shows how to create an item with a single input artifact:

```python
InputArtifactCreationRequest(
    name="user_slide", # as defined by the application version input_artifact schema
    download_url="<a signed url to download the data>",
    metadata={
        "checksum_crc32c": "<checksum>",
        "base_mpp": "<base_mpp>",
        "width": "<width>",
        "height": "<height>"
    }
)
```

In [3]:
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/Users/akunft/Downloads/aignx-platform-api-shsodcule-9086ce65109a.json"

from aignx.codegen.models import (
    ApplicationVersion,
    InputArtifactCreationRequest,
    ItemCreationRequest,
    RunCreationRequest,
)

from aignostics.client.utils import generate_signed_url

payload = [
    ItemCreationRequest(
        reference="1",
        input_artifacts=[
            InputArtifactCreationRequest(
                name="user_slide",
                download_url=generate_signed_url(
                    "gs://aignx-storage-service-dev/sample_data_formatted/9375e3ed-28d2-4cf3-9fb9-8df9d11a6627.tiff"
                ),
                metadata={
                    "checksum_crc32c": "N+LWCg==",
                    "base_mpp": 0.46499982,
                    "width": 3728,
                    "height": 3640,
                },
            )
        ],
    ),
]

application_run = client.runs.create(
    RunCreationRequest(
        application_version=ApplicationVersion("60e7b441-307a-4b41-8a97-5b02e7bc73a4"),
        items=payload,
    )
)
print(application_run)

Application run `d51def04-326b-4138-a411-47bc073bc5ac`: running, 1 items - (1/0/0) [pending/succeeded/error]


# Observe the status of the application run and download

While you can observe the status of an application run directly via the `status()` method and also retrieve the results via the `results()` method, you can also download the results directly to a folder of your choice. The `download_to_folder()` method will download all the results to the specified folder. The method will automatically create a sub-folder in the specified folder with the name of the application run. The results for each individual input item will be stored in a separate folder named after the `reference` you defined in the `ItemCreationRequest`.

The method downloads the results for a slide as soon as they are available. There is no need to keep the method running until all results are available. The method will automatically check for the status of the application run and download the results as soon as they are available. If you invoke the method on a run you already downloaded some results before, it will only download the missing artifacts.

In [ ]:
import tempfile

download_folder = tempfile.gettempdir()
application_run.download_to_folder(download_folder)

# Continue to retrieve results for an application run

In case you just triggered an application run and want to check on the results later or you had a connection loss, you can simply initialize an applicaiton run object via it's `application_run_id`. If you do not have the `application_run_id` anymore, you can simple list all currently running application version via the `client.runs.list()` method. The `application_run_id` is part of the `ApplicationRun` object returned by the `list()` method. You can then use the `download_to_folder()` method to continue downloading the results.

In [4]:
# list currently running applications
application_runs = client.runs.list()
for run in application_runs:
    print(run)

Application run `01a5cb5b-1c34-48fb-be83-c6fa886f8c20`: completed, 3 items - (0/3/0) [pending/succeeded/error]
Application run `026a1c42-537b-455c-9459-abb0ee9970eb`: completed, 3 items - (0/3/0) [pending/succeeded/error]
Application run `029ec139-46b7-49d6-ae70-29aea43dc757`: completed, 3 items - (0/3/0) [pending/succeeded/error]
Application run `02f85458-76ca-4400-863a-8b7cd55bc7e5`: completed, 3 items - (0/3/0) [pending/succeeded/error]
Application run `04c915b8-3b2a-4b7a-be76-7766190ca60d`: completed, 3 items - (0/3/0) [pending/succeeded/error]
Application run `06434c6c-eb57-436b-bdf7-82bede577d8b`: running, 3 items - (1/0/2) [pending/succeeded/error]
Application run `0a543af7-3b72-4ff0-80bd-6aab58571969`: completed, 3 items - (0/3/0) [pending/succeeded/error]


KeyboardInterrupt: 

In [7]:
import tempfile

from aignostics.client.resources.runs import ApplicationRun

application_run = ApplicationRun.for_application_run_id("d51def04-326b-4138-a411-47bc073bc5ac")
# download

download_folder = tempfile.gettempdir()
application_run.download_to_folder(download_folder)

> Download for tissue_segmentation:geojson_polygons to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1/tissue_segmentation:geojson_polygons.json


  0%|          | 0.00/2.00 [00:00<?, ?B/s]

> Download for tissue_segmentation:tiff_heatmap to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1/tissue_segmentation:tiff_heatmap.tiff


  0%|          | 0.00/4.02M [00:00<?, ?B/s]

> Download for tissue_qc:tiff_heatmap to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1/tissue_qc:tiff_heatmap.tiff


...

> Download for tissue_qc:geojson_polygons to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1/tissue_qc:geojson_polygons.json


  0%|          | 0.00/2.00 [00:00<?, ?B/s]

> Download for tissue_segmentation:csv_class_information to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1/tissue_segmentation:csv_class_information.csv


  0%|          | 0.00/103 [00:00<?, ?B/s]

Downloaded results for item: 1 to /var/folders/65/t33k_dcn45s6545hgkwg9y3m0000gn/T/d51def04-326b-4138-a411-47bc073bc5ac/1
